In [3]:
import os

In [4]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow/research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow'

In [7]:
from dataclasses import dataclass
from pathlib import Path
from mlProject import logger

In [8]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [9]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories 

In [14]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/ntchinda1998/recomProject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="ntchinda1998"
os.environ["MLFLOW_TRACKING_PASSWORD"]="7844793f4d32031055b8edc36b79ffd8c715e50c"

In [10]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH
            ) -> None:
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:

        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri = "https://dagshub.com/ntchinda1998/recomProject.mlflow"
        )

        return model_evaluation_config

In [16]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [ ]:
from abc import ABC, abstractmethod
import numpy as np

class ModelEvaluator(ABC):
    @abstractmethod
    def eval_metrics(self, X_user_val: np.ndarray, X_movie_val: np.ndarray, y_rating_val: np.ndarray, model: RecommenderTrainer) -> Tuple[float, float]:
        pass

    @abstractmethod
    def log_into_mlflow(self) -> None:
        pass

In [17]:
from mlProject.utils.common import save_json
from mlProject.components.model_trainer import RecommenderTrainer
from typing import Annotated, Tuple
from mlProject.entity.config_entity import ModelTrainerConfig


class RecommendModeEvaluator(ModelEvaluator):
    def __init__(self, config: ModelTrainerConfig) -> None:

        self.config = config
    
    def eval_metrics(self, X_user_val: np.ndarray, X_movie_val: np.ndarray, y_rating_val: np.ndarray, model: RecommenderTrainer) -> Tuple[
        Annotated[float, "MAE"], 
        Annotated[float, "MSE"]
    ]:

        mae, mse = model.evaluate([X_user_val, X_movie_val], y_rating_val)

        return mae, mse
    
    def log_into_mlflow(self, X_user_val: np.ndarray, X_movie_val: np.ndarray, y_rating_val: np.ndarray, model: RecommenderTrainer):

        with mlflow.start_run():

            (mae, mse) = self.eval_metrics(X_user_val: np.ndarray, X_movie_val: np.ndarray, y_rating_val: np.ndarray, model: RecommenderTrainer)

            hyperparameters = {
                "learning_rate": config.learning_rate,
                "validation_split": config.validation_split,
                "epochs": config.epochs,
                "batch_size": config.batch_size
            }
            mlflow.log_params(hyperparameters)

            mlflow.log_metric("mse", mse)
            mlflow.log_metric("mae", mae)


In [19]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2024-08-19 06:34:40,507: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2024-08-19 06:34:40,535: INFO: common: Yaml file : params.yaml loaded successfully]
[2024-08-19 06:34:40,558: INFO: common: Yaml file : schema.yaml loaded successfully]
[2024-08-19 06:34:40,564: INFO: common: Created directory at: artifacts]
[2024-08-19 06:34:40,580: INFO: common: Created directory at: artifacts/model_evaluation]


[2024-08-19 06:34:41,435: INFO: common: Json file saved at: artifacts/model_evaluation/metrics.json]


Registered model 'ElasticNet' already exists. Creating a new version of this model...
2024/08/19 06:35:23 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticNet, version 2
Created version '2' of model 'ElasticNet'.


In [ ]:
import dagshub
dagshub.init(repo_owner='ntchinda1998', repo_name='recomProject', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

In [ ]:
"https://dagshub.com/ntchinda1998/recomProject.mlflow"

In [ ]:
t 

In [ ]:
impor

In [ ]:
from datetime import datetime
from typing import List

from zenml import pipeline, step
from zenml.config import DockerSettings
from zenml.constants import DEFAULT_SERVICE_START_STOP_TIMEOUT
from zenml.integrations.constants import MLFLOW
from zenml.integrations.mlflow.steps import mlflow_model_deployer_step
from zenml.steps import BaseParameters, Output

# Parameters for model evaluation
class EvaluationParameters(BaseParameters):
    evaluation_metric: str = "accuracy"
    minimum_threshold: float = 0.75

# Step to fetch and compare model metrics
@step
def model_evaluation_step(
    current_model_metrics: dict,
    evaluation_params: EvaluationParameters,
) -> bool:
    """Compare current model with existing deployed models"""
    from zenml.services import BaseService
    from zenml.client import Client

    client = Client()
    model_deployer = client.active_stack.model_deployer

    # Get all running services
    existing_services = model_deployer.find_model_server(
        running=True,
        timeout=DEFAULT_SERVICE_START_STOP_TIMEOUT
    )

    current_metric = current_model_metrics[evaluation_params.evaluation_metric]
    should_deploy = True

    for service in existing_services:
        deployed_model_metrics = service.get_metrics()
        if deployed_model_metrics[evaluation_params.evaluation_metric] >= current_metric:
            should_deploy = False
            break

    return should_deploy and current_metric >= evaluation_params.minimum_threshold

# Your existing training step
@step
def train_model() -> tuple:
    """Your existing training logic"""
    # Your training code here
    return model, metrics

# Pipeline definition
@pipeline(enable_cache=False, settings={"docker": DockerSettings(required_integrations=[MLFLOW])})
def training_pipeline():
    """Main training pipeline with evaluation and conditional deployment"""
    model, metrics = train_model()
    should_deploy = model_evaluation_step(metrics, EvaluationParameters())
    
    if should_deploy:
        mlflow_model_deployer_step(
            model=model,
            deploy_decision=should_deploy
        )

# Schedule configuration
from zenml.client import Client
from zenml.config.schedule import Schedule

client = Client()

# Create schedule (runs every day at 2 AM)
schedule = Schedule(
    cron_expression="0 2 * * *",  # Cron expression for daily at 2 AM
    pipeline=training_pipeline,
    enable_running=True,
)

# Register the schedule
client.create_schedule(schedule)